# 証券営業インテリジェンス ハンズオン
## Part 5: Cortex Agent（Snowflake Intelligence）

このノートブックでは、これまでに作成したすべてのサービスを統合した **Cortex Agent** を作成します。
作成したエージェントは **Snowflake Intelligence** の UI から自然言語で対話できます。

### Cortex Agent とは

**Cortex Analyst** + **Cortex Search** + **AI** を組み合わせた、マルチツール対話型AIアシスタントです。

```
ユーザーからの質問
      │
Cortex Agent（オーケストレーター）
  ├── 顧客・資産データが必要    → Cortex Analyst（CUSTOMER_WEALTH_SEMANTIC_VIEW）
  ├── 信託商品データが必要      → Cortex Analyst（TRUST_PRODUCT_SEMANTIC_VIEW）
  ├── ニュースが必要            → Cortex Search（NEWS_SEARCH_SERVICE）
  ├── 商品説明書が必要          → Cortex Search（LOAN_DOCS_SEARCH_SERVICE）
  ├── アナリストレポートが必要   → Cortex Search（ANALYST_REPORT_SEARCH_SERVICE）
  └── ファンド目論見書が必要     → Cortex Search（FUND_DOCS_SEARCH_SERVICE）
      │
  複数ツールの結果を統合して回答を生成
```

### 体験ポイント: Snowflake にデータを集めるだけで AI が動く！

**2種類のデータ形式に対応しており、計6ツールを統合しています。**

| データ種別 | ツール種類 | 統合データ | 答えられる質問の例 |
|---|---|---|---|
| **構造化データ（DB）** | Cortex Analyst | 顧客資産・ポートフォリオ・相続税試算 | C001の含み益は？相続税が高い顧客は？ |
| **構造化データ（DB）** | Cortex Analyst | 信託商品条件・スキーム | 証券担保ローンの金利条件は？ |
| **非構造化データ（文書）** | Cortex Search | マーケットニュース | 今週の重要ニュースは？ |
| **非構造化データ（文書）** | Cortex Search | ローン・信託商品説明書 | 遺言信託の手続きは？ |
| **非構造化データ（文書）** | Cortex Search | アナリストレポート | トヨタのアナリスト評価は？ |
| **非構造化データ（文書）** | Cortex Search | ファンド目論見書 | 野村AMファンドの信託報酬は？ |

> **💡 ポイント** Snowflake にデータを集めるだけで、Cortex Analyst（構造化クエリ）と Cortex Search（文書検索）を通じて AI が自動的に最適なツールを選択します。追加のインフラは不要です。

### 前提条件

- `part3_cortex_analyst.ipynb` 実行済み（Semantic View 2つ）
- `part4_cortex_search.ipynb` 実行済み（Cortex Search Service 4つ）

> ⏱️ **このパートの目安時間: 40分**


In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

images_dir = 'images/'

def display_image(image_file: str) -> None:
    image_path = os.path.join(images_dir, image_file)
    img = Image.open(image_path)
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [ ]:
display_image('cortex-agent.png')

In [ ]:
%%sql -r result_env
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA, CURRENT_WAREHOUSE() AS WH;

## 1. 前提条件の確認

Cortex Agent を作成する前に、必要なオブジェクトがすべて揃っているか確認します。

> ⏱️ **目安: 3分**


In [ ]:
%%sql -r result_check_sv
-- Semantic View の確認
SHOW SEMANTIC VIEWS IN SCHEMA DEMO_SCHEMA;

In [ ]:
%%sql -r result_check_cs
-- Cortex Search Service の確認
-- READY ステータスになっていることを確認する
SHOW CORTEX SEARCH SERVICES IN SCHEMA DEMO_SCHEMA;

## 2. 全ツール統合版エージェントの作成

Cortex Search と Cortex Analyst を統合した完全版エージェントを SQL で一括作成します。

> **💡 注意**: このセルを実行すると、エージェントが作成/更新されます。
> 「サンプル質問」の設定も含まれています。

> ⏱️ **目安: 5分**


In [ ]:
%%sql -r result_agent_full
-- ============================================================
-- Cortex Agent 完全版（6ツール + サンプル質問）
-- 構造化データ: Cortex Analyst x2（顧客資産DB・信託商品DB）
-- 非構造化データ: Cortex Search x4（ニュース・商品説明書・アナリストレポート・ファンド目論見書）
-- ============================================================
CREATE OR REPLACE AGENT WEALTH_MANAGEMENT_ASSISTANT_AGENT
WITH PROFILE = '{"display_name": "ウェルスマネジメントアシスタント"}'
COMMENT = '証券営業担当者向け総合支援AIアシスタント。顧客情報・ポートフォリオ分析・信託商品提案・市場ニュース・アナリストレポートを統合して最適な提案を支援します。'
FROM SPECIFICATION $$
{
    "models": {
        "orchestration": ""
    },
    "instructions": {
        "response": "あなたは証券営業担当者を支援するアシスタントです。\n\n【基本原則】\n・質問に直接回答し、求められていない情報は出さない\n・ツールで取得した情報のみを使用（推測禁止）\n・推奨は必ず前提条件付きで提示\n・不明点は「確認事項」として明示\n\n【出力形式】\nデフォルト：社内メモ（RM向け）\n顧客向けトーク：ユーザーが明示した場合のみ\n\n【個別顧客相談の構造】\n1. 確認事項：支払期限、取得単価、流動資産内訳、担保余力\n2. 現状サマリ：保有銘柄、時価、含み益\n3. 市況（必要時）：出典を明記\n4. 選択肢比較（表形式）\n5. 推奨（前提条件付き）＋リスク\n6. 次アクション\n\n【税金表示のルール】\n・取得単価不明時は税額を確定値で出さない\n・数値提示時は必ず：前提（税率/含み益）＋レンジ表示を付記\n\n【禁止事項】\n・取得単価未確認での税額確定値提示\n・必要資金に対する過剰売却・過剰借入の推奨",
        "orchestration": "【ツール選択原則】\n・質問に必要なツールのみ使用\n・PII（個人情報）は最小限、一覧では匿名化\n\n【ツール種別の理解】\n・Cortex Analyst（text-to-SQL）: 構造化DBへのクエリ。数値集計・絞り込み・ランキングが得意\n・Cortex Search（意味検索）: ドキュメント・テキストからの情報取得。説明・条件文・ニュース記事が得意\n\n【ツール選択ルール（単体）】\n・顧客リスト/検索/資産照会/相続税試算 → CUSTOMER_WEALTH_ANALYST\n・証券担保ローン/信託商品の金利・金額・数値条件 → TRUST_PRODUCT_ANALYST\n・証券担保ローン/信託商品の仕組み・手続き・制度説明 → LOAN_DOCS_SEARCH\n・マーケットニュース・市場動向 → NEWS_SEARCH\n・株式アナリストレポート・投資判断 → ANALYST_REPORT_SEARCH\n・ファンド目論見書・信託報酬・分配金 → FUND_DOCS_SEARCH\n\n【ツール選択ルール（複合）】\n・売却検討＋市場評価 → CUSTOMER_WEALTH_ANALYST + NEWS_SEARCH + ANALYST_REPORT_SEARCH\n・相続対策＋商品提案 → CUSTOMER_WEALTH_ANALYST + TRUST_PRODUCT_ANALYST + LOAN_DOCS_SEARCH\n・顧客資産＋ファンド照会 → CUSTOMER_WEALTH_ANALYST + FUND_DOCS_SEARCH\n・銘柄ニュース＋アナリスト評価 → NEWS_SEARCH + ANALYST_REPORT_SEARCH"
    },
    "tools": [
        {
            "tool_spec": {
                "type": "cortex_analyst_text_to_sql",
                "name": "CUSTOMER_WEALTH_ANALYST",
                "description": "【構造化DB】顧客マスタ・ポートフォリオ・取引履歴・相続税試算のデータベースを自然言語クエリで分析する。顧客検索・資産照会・含み益確認・相続税試算・ライフイベント確認に使用する"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_analyst_text_to_sql",
                "name": "TRUST_PRODUCT_ANALYST",
                "description": "【構造化DB】信託銀行商品（証券担保ローン・教育資金贈与信託・遺言信託等）の条件・税制メリット・推奨条件をデータベースから検索する"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "NEWS_SEARCH",
                "description": "【文書検索】マーケットニュースをセマンティック検索する。銘柄動向・経済指標・税制改正・市場トレンドなど、最新ニュース記事の取得に使用する"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "LOAN_DOCS_SEARCH",
                "description": "【文書検索】ローン・信託商品の説明書をセマンティック検索する。証券担保ローン・教育資金贈与信託・遺言信託の制度説明・手続き・注意事項の取得に使用する"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "ANALYST_REPORT_SEARCH",
                "description": "【文書検索】株式アナリストレポートをセマンティック検索する。投資判断（買い/中立/売り）・目標株価・業績見通し・リスク要因の取得に使用する"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "FUND_DOCS_SEARCH",
                "description": "【文書検索】野村AMファンドの目論見書をセマンティック検索する。ファンドの特色・投資方針・信託報酬・分配金方針・投資リスクの取得に使用する"
            }
        }
    ],
    "tool_resources": {
        "CUSTOMER_WEALTH_ANALYST": {
            "semantic_view": "SNOWFINANCE_DB.DEMO_SCHEMA.CUSTOMER_WEALTH_SEMANTIC_VIEW"
        },
        "TRUST_PRODUCT_ANALYST": {
            "semantic_view": "SNOWFINANCE_DB.DEMO_SCHEMA.TRUST_PRODUCT_SEMANTIC_VIEW"
        },
        "NEWS_SEARCH": {
            "name": "SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE",
            "max_results": 5,
            "title_column": "TITLE",
            "id_column": "NEWS_ID"
        },
        "LOAN_DOCS_SEARCH": {
            "name": "SNOWFINANCE_DB.DEMO_SCHEMA.LOAN_DOCS_SEARCH_SERVICE",
            "max_results": 5,
            "title_column": "TITLE",
            "id_column": "DOC_ID"
        },
        "FUND_DOCS_SEARCH": {
            "name": "SNOWFINANCE_DB.DEMO_SCHEMA.FUND_DOCS_SEARCH_SERVICE",
            "max_results": 5,
            "title_column": "SECTION_HEADER",
            "id_column": "FILE_NAME"
        },
        "ANALYST_REPORT_SEARCH": {
            "name": "SNOWFINANCE_DB.DEMO_SCHEMA.ANALYST_REPORT_SEARCH_SERVICE",
            "max_results": 5,
            "title_column": "REPORT_TITLE",
            "id_column": "REPORT_ID"
        }
    },
    "sample_questions": [
        "C001の山田太郎様のポートフォリオと含み益を教えてください",
        "相続税の試算が高い顧客を上位10名リストアップしてください",
        "トヨタ株の最新アナリスト評価と投資判断を教えてください",
        "今週の重要なマーケットニュースを3件教えてください",
        "証券担保ローンの条件と金利を教えてください",
        "教育資金贈与信託の制度について、期限も含めて教えてください",
        "野村AMのファンドの信託報酬と分配金方針を教えてください"
    ]
}
$$;
SELECT 'Cortex Agent（完全版: 6ツール）作成完了！' AS STATUS;

In [ ]:
%%sql -r result_describe_agent
-- エージェントの詳細確認（追加されたツール一覧）
DESCRIBE AGENT SNOWFINANCE_DB.DEMO_SCHEMA.WEALTH_MANAGEMENT_ASSISTANT_AGENT;

## 3. Snowflake Intelligence への公開と動作確認

### Snowflake Intelligence へのアクセス手順

1. **Snowsight** にログイン
2. 左メニュー「**Snowflake Intelligence**」をクリック
3. 「**+ New Conversation**」をクリック
4. エージェント選択画面で `WEALTH_MANAGEMENT_ASSISTANT_AGENT` を選択
5. チャットを開始！

### まず基本動作を確認しましょう

```
C001の山田太郎様について教えてください
```

エージェントが複数ツールを呼び出して、顧客情報・ポートフォリオ・関連ニュースを統合した回答を返すことを確認します。

### AIがどのツールを使っているか確認しよう

Snowflake Intelligence では、エージェントが「なぜこの回答をしたか」を可視化できます。

1. 回答の下部にある **「Sources」** または **「使用ツール」** をクリック
2. どの Cortex Analyst / Cortex Search が呼び出されたかが表示されます
3. Cortex Analyst の場合は **生成されたSQL** も確認できます

**確認ポイント（質問と期待されるツール呼び出し）**:

| 質問例 | 呼ばれるツール | データの種類 |
|---|---|---|
| C001の山田様の資産は？ | `CUSTOMER_WEALTH_ANALYST` | 構造化DB → text-to-SQL |
| トヨタ株のニュースは？ | `NEWS_SEARCH` | 非構造化文書 → 意味検索 |
| 証券担保ローンの条件は？ | `LOAN_DOCS_SEARCH` | 非構造化文書 → 意味検索 |
| 山田様の保有株＋最新評価は？ | `CUSTOMER_WEALTH_ANALYST` + `ANALYST_REPORT_SEARCH` | **複数ツール同時呼び出し** |

> **💡 ポイント** 複数ツールが同時に呼ばれる質問こそ、Cortex Agent の真価が発揮される瞬間です。Snowflake に置いたデータが、AIを通じて統合的に回答されていることを確認してください。

> ⏱️ **目安: 10分**（動作確認）


## 4. デモシナリオ: 山田様の相談

### 背景設定

```
顧客: 山田太郎様（68歳・元上場企業役員）
預かり資産: 約8億円
相談内容: 「孫の海外留学費用として2,000万円が必要。
           トヨタ株を売ろうかと思っている。」
```

---

### 🎯 Step 1: 顧客情報の確認

Snowflake Intelligence に以下を入力してみましょう：

```
C001の山田太郎様のプロフィールとポートフォリオを教えてください。
特にトヨタ株（7203）の保有状況と含み益を確認したいです。
```

**期待される回答**: 
- 山田様の基本情報（年齢・資産・担当RM）
- トヨタ株の保有株数・時価・含み益の金額
- 関連するライフイベント（孫の教育資金）

---

### 🎯 Step 2: 売却 vs ローンの比較提案

```
山田様がトヨタ株を売って2,000万円を確保しようとしています。
税金面でのデメリットと、証券担保ローンを使った場合との比較を教えてください。
```

**期待される回答（AIが複数ツールを同時使用）**:
- **Cortex Analyst**: 山田様のトヨタ株含み益 → 譲渡税の試算
- **LOAN_DOCS_SEARCH**: 証券担保ローンの金利・条件
- **ANALYST_REPORT_SEARCH**: トヨタ株の直近のアナリスト評価
- 比較表形式でのまとめ

---

### 🎯 Step 3: 長期的な相続・贈与対策

```
山田様の相続税の試算と、長期的な観点から教育資金贈与信託が有効かどうか教えてください。
制度の期限も含めてお願いします。
```

**期待される回答（さらに深い提案）**:
- **Cortex Analyst (INHERITANCE_TAX)**: 相続税試算額
- **TRUST_PRODUCT_ANALYST**: 教育資金贈与信託の条件・期限・税制メリット
- 短期対応（証券担保ローン）と長期対応（相続対策）の統合提案

---

### 🎯 Step 4: 訪問準備資料の自動作成

```
明日の山田様との訪問に向けて、提案書のドラフトを作成してください。
証券担保ローンと教育資金贈与信託の両方を盛り込んでください。
```

> ⏱️ **目安: 15分**（各質問をSnowflake Intelligenceに入力して体験）


## 5. よくある質問例（応用練習）

以下の質問を Snowflake Intelligence で試してみましょう。
各質問の横に「使用ツール」を記載しています。**どのデータが使われているか意識しながら試してください。**

### 顧客検索・資産分析
**使用ツール: `CUSTOMER_WEALTH_ANALYST`（Cortex Analyst - 顧客資産DB）**
```
預かり資産5億円以上で、相続対策に関心がありそうな顧客をリストアップしてください

担当RM001の顧客で、最近30日間コンタクトしていない顧客を教えてください

トヨタ株を保有している顧客の含み益ランキングトップ10を出してください
```

### 銘柄・マーケット情報
**使用ツール: `ANALYST_REPORT_SEARCH`（Cortex Search - アナリストレポート）+ `NEWS_SEARCH`（Cortex Search - ニュース）**
```
トヨタ株の最新のアナリスト評価を教えてください

今週の市場ニュースで、特に重要なものを3つ教えてください
```

### 商品・制度照会
**使用ツール: `LOAN_DOCS_SEARCH`（Cortex Search - 商品説明書）+ `TRUST_PRODUCT_ANALYST`（Cortex Analyst - 信託商品DB）**
```
教育資金贈与信託の制度について、期限も含めて詳しく教えてください

証券担保ローンの担保率と金利条件を教えてください
```

### ファンド照会
**使用ツール: `FUND_DOCS_SEARCH`（Cortex Search - ファンド目論見書）**
```
野村AMのファンドの中で信託報酬が低いものを教えてください

分配金方針が毎月分配型のファンドはどれですか？
```

### 相続税・試算
**使用ツール: `CUSTOMER_WEALTH_ANALYST`（Cortex Analyst）+ `TRUST_PRODUCT_ANALYST`（Cortex Analyst）**
```
相続税が1億円を超える顧客は何人いますか？年齢帯別に集計してください

相続税対策として有効な信託商品はどれですか？条件も含めて教えてください
```

### マルチツール統合推論（最も難しい質問）
**使用ツール: 複数ツールを同時使用**
```
来週の訪問リストに向けて、証券担保ローンの提案が特に有効な顧客を5名選んでください。
選定理由と各顧客への提案アプローチも添えてください。
```
> **💡 確認ポイント** Sources をクリックして、複数のツールが呼び出されていることを確認しましょう。


## 6. エージェントのモニタリング

エージェントの動作を監視して、精度向上に役立てます。

### モニタリング手順

1. Snowsight → 「**AI と ML**」→「**エージェント**」を開く
2. `WEALTH_MANAGEMENT_ASSISTANT_AGENT` を選択
3. 「**分析**」タブで以下を確認：
   - **質問数・回答数**: 利用状況の把握
   - **ツール呼び出し頻度**: どのツールが多く使われているか
   - **レイテンシ**: 応答速度の監視
   - **Thumbs Up/Down**: ユーザーフィードバック

### パフォーマンス改善のポイント

- ⬆️ **Cortex Search の精度向上**: データ量を増やす、セクション分割を細かくする
- ⬆️ **Semantic View の充実**: 同義語・メトリクス定義を追加する
- ⬆️ **エージェント instruction の改善**: 回答形式・ツール選択ルールを調整


In [ ]:
%%sql -r result_final_tables
-- 作成されたすべてのオブジェクトの最終確認
SELECT 'TABLE' AS TYPE, TABLE_NAME AS OBJECT_NAME, '' AS STATUS
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'DEMO_SCHEMA' AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TYPE, OBJECT_NAME;

In [ ]:
%%sql -r result_final_services
-- サービスとエージェントの確認
SHOW CORTEX SEARCH SERVICES IN SCHEMA DEMO_SCHEMA;

In [ ]:
%%sql -r result_final_sv
SHOW SEMANTIC VIEWS IN SCHEMA DEMO_SCHEMA;

In [ ]:
%%sql -r result_final_agents
SHOW AGENTS IN SCHEMA DEMO_SCHEMA;

## まとめ

ハンズオン完了！4時間で証券営業インテリジェンスシステムを構築しました。

### 作成したオブジェクト一覧

| コンポーネント | オブジェクト名 | 役割 |
|---|---|---|
| **Cortex AI Functions** | AI_COMPLETE, AI_SENTIMENT, AI_EXTRACT, AI_CLASSIFY, AI_REDACT, AI_SIMILARITY, AI_PARSE_DOCUMENT | テキスト分析・生成 |
| **Cortex Analyst** | CUSTOMER_WEALTH_SEMANTIC_VIEW | 顧客資産データの自然言語クエリ |
| **Cortex Analyst** | TRUST_PRODUCT_SEMANTIC_VIEW | 信託商品の自然言語クエリ |
| **Cortex Search** | NEWS_SEARCH_SERVICE | ニュースのセマンティック検索 |
| **Cortex Search** | LOAN_DOCS_SEARCH_SERVICE | 商品説明書のセマンティック検索 |
| **Cortex Search** | ANALYST_REPORT_SEARCH_SERVICE | アナリストレポートのセマンティック検索 |
| **Cortex Search** | FUND_DOCS_SEARCH_SERVICE | ファンド目論見書のセマンティック検索 |
| **Cortex Agent** | WEALTH_MANAGEMENT_ASSISTANT_AGENT | 統合AIアシスタント |

### 4つの体験ポイント

1. **AI Functions**: 「2,000文字のレポートが3行に。1時間の準備が10秒に」
2. **Cortex Analyst**: 「SQLを書かずに日本語で顧客データを分析できる」
3. **Cortex Search**: 「証券担保ローンを知らなくても、意味を理解して発見できる」
4. **Cortex Agent**: 「データが増えるほど、AIの回答が賢くなる」

### 次のステップ

このハンズオンで作成したシステムを実際の業務データに適用する場合：

1. **実データの投入**: 顧客マスタ・取引履歴を実データに置き換え
2. **セキュリティ設定**: Row Access Policy で担当RM別にデータアクセス制限
3. **定期更新**: Cortex Search の TARGET_LAG をリアルタイムに近づける
4. **フィードバックループ**: エージェントのモニタリングで継続的改善

### ありがとうございました！

Snowflake Cortex AI と Snowflake Intelligence を活用することで、
**データドリブンな証券営業**が実現できます。
疑問点・ご要望はお気軽にお声がけください。
